# 34 — Target-footprint diagnostic: what predicts per-target GBSA BEDROC?

## A univariate signal-vs-noise audit of the 77 MD/ligand-chem features

> **Reader's guide — what this notebook does, in plain language**
>
> **Question:** Instead of throwing all 77 features into an ML model and hoping it
> finds signal (NB 23-30), can we identify — one feature at a time — which values
> actually correlate with per-target GBSA-locked BEDROC? A robust signal here means
> a physically-interpretable predictor: given a new target's simple properties, we
> can guess a priori whether MMGBSA will rank its ligands well.
>
> **Three questions, three tests:**
> - **Q1 — Protein-side aggregate → BEDROC.** Per target, take the median of each
>   protein-side feature (backbone RMSD, active-site RMSF, protein Rg, active-site
>   formal charge, …). Correlate with per-target BEDROC. Physical intuition:
>   flexible / charged pockets are easier for GBSA to discriminate.
> - **Q2 — Ligand-side aggregate → BEDROC.** Same setup but for ligand-side features
>   (drift, escape fraction, buried SASA, dipole, …). Physical intuition: probably
>   none — a *median across 30 different ligands* smooths away most target-specific
>   signal.
> - **Q3 — binder-vs-non-binder feature separation → BEDROC.** Per target, how well
>   does a feature discriminate labelled binders from labelled non-binders? Measured
>   *two ways* to avoid a scale-sensitive artifact:
>   - **Cohen d** = (mean binder − mean non-binder) / pooled std. Parametric,
>     inflated by outliers.
>   - **Cliff's delta** = P(binder > non-binder) − P(binder < non-binder). Rank-based,
>     bounded, robust.
>
> **Vocabulary:** we call inactives *labelled non-binders* / *measured inactives*
> (pchembl < 5, experimentally measured), never *decoys* — the term "decoy" is
> reserved for computationally-generated fake inactives (DEKOIS, DUD-E), which this
> dataset does not contain.
>
> **How to read the numbers:**
> - `ρ` = Spearman rank correlation across the 8 labelled targets, values in [−1, 1]
> - `p` = two-sided Spearman p-value (no multiple-testing correction inside this cell)
> - `Cohen d` and `Cliff's δ` reported side by side — divergence between them signals
>   distribution-sensitive noise
>
> **Bottom line preview:** Q1 (protein flexibility) shows real signal (ρ = +0.88,
> p = 0.004, robust to metric choice). Q2 (ligand aggregates) is noise. Q3 looks
> strong with Cohen d (ρ = +0.88) but collapses under Cliff's δ (ρ = +0.43,
> non-significant) → the parametric d was outlier-inflated. Only Q1 survives as a
> deployable diagnostic.

> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-feature analysis and downstream ranking questions.
>
> See STUDY_DESIGN Chapter §A3 Q1 (per-target combo selection) and Q2 (single-feature panel
> ranker) for the framing this notebook addresses.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` +
> `data/raw/reference/ohds_metadata.csv` (and `data/derived/canonical_baselines.csv` for
> baseline comparison).

In [ ]:
# --- notebook preamble (same pattern as NB 22-32) ---
NB_STEM = "45_target_footprint_diagnostic"

import sys, os
from pathlib import Path

# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.linear_model import LinearRegression
from IPython.display import display

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM
from discovery9.paths import ROOT, DERIVED, FIGURES, GBSA_STUDY
from discovery9.io import load_features, load_metadata
from discovery9.metrics import bedroc
apply_style()

# --- fig-capture hook so the last cell exports every figure ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS: _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS: _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

In [ ]:
# ===== Load data and compute per-target GBSA-locked BEDROC =====
COMBO = "igb2_di4_salt0.15_st0.0072"   # reviewer-locked baseline combo

meta = load_metadata()
gbsa = pd.read_csv(GBSA_STUDY / 'data' / 'raw' / 'gbsa_dG_raw.csv')
gbsa = gbsa[gbsa.combo == COMBO][['complex_id','target','mean_dG_kcalmol']].rename(
    columns={'mean_dG_kcalmol': 'gbsa_dG'})
feat = load_features(with_ligand_chem=True)

data = (meta.merge(gbsa, on=['complex_id','target'], how='inner')
             .merge(feat, on=['complex_id','target'], how='inner', suffixes=('','_dup'))
             .dropna(subset=['is_active','gbsa_dG']))
data['gbsa_score'] = -data.gbsa_dG   # higher score = better predicted binder

tgt_bedroc = pd.Series({t: bedroc(g.gbsa_score.values, g.is_active.astype(int).values)
                        for t, g in data.groupby('target')})
tgt_bedroc = tgt_bedroc.sort_values(ascending=False)

print(f'Labelled panel: {len(data)} complexes, {data.target.nunique()} targets')
print(f'\nPer-target BEDROC α=20 of GBSA-locked (our regression target):')
print(tgt_bedroc.round(3).to_string())

## Q1 — Protein-side aggregate features vs per-target BEDROC

For each protein-side feature `f`, take the median across the 30 ligands per target and
run Spearman rank correlation against per-target BEDROC of GBSA-locked. Protein-side
features are target properties in their own right (they describe the receptor, not the
specific ligand), so they cannot be a molecular-weight confound of the labelled ligand set.

In [ ]:
# Split features into protein-side vs ligand-side by name prefix.
FEATS_ALL = [c for c in feat.columns
             if c not in {'complex_id','target','pchembl','is_active','ligand_file','source'}
             and feat[c].dtype in (np.float64, np.int64)]

LIG = [c for c in FEATS_ALL if (c.startswith('lig_') or c.startswith('n_hb') or c.startswith('hb_')
                                 or c.startswith('vdw_') or c.startswith('ifp_')
                                 or c.startswith('salt_bridges'))]
PROT = [c for c in FEATS_ALL if c not in LIG]

per_t_med = data.groupby('target')[FEATS_ALL].median()
per_t_med = per_t_med.loc[tgt_bedroc.index.intersection(per_t_med.index)]
y = tgt_bedroc.reindex(per_t_med.index).values

def corr_table(features):
    rows = []
    for f in features:
        x = per_t_med[f].values
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 5 or np.std(x[m]) == 0: continue
        rho, p = spearmanr(x[m], y[m])
        rows.append({'feature': f, 'rho': rho, 'p': p})
    return pd.DataFrame(rows).sort_values('rho', key=abs, ascending=False).reset_index(drop=True)

prot_corr = corr_table(PROT)
lig_corr = corr_table(LIG)

print(f'PROTEIN-side features: {len(PROT)}   ·   {(prot_corr.p<0.05).sum()} with p<0.05')
print(f'LIGAND-side features:  {len(LIG)}   ·   {(lig_corr.p<0.05).sum()} with p<0.05')
print()
print('=== Q1 · TOP 15 protein-side features (sorted by |ρ|) ===')
display(prot_corr.head(15).round(3))

# Save derived table
prot_corr.to_csv(DERIVED / 'target_footprint_q1_protein.csv', index=False)
lig_corr.to_csv(DERIVED / 'target_footprint_q2_ligand.csv', index=False)

In [ ]:
# Figure 1 — Q1 protein-side + Q2 ligand-side top-features horizontal bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, tab, title, primary in [
        (axes[0], prot_corr.head(12).iloc[::-1], 'Q1 · PROTEIN-side (median per target)', NAVY),
        (axes[1], lig_corr.head(12).iloc[::-1],  'Q2 · LIGAND-side (median per target)',  GOLD)]:
    ys = np.arange(len(tab))
    colors = [primary if p < 0.05 else GREYD for p in tab['p']]
    ax.barh(ys, tab['rho'], color=colors, edgecolor=NAVY, linewidth=0.5)
    ax.set_yticks(ys)
    ax.set_yticklabels(tab['feature'], fontsize=8)
    ax.axvline(0, color=GREY, lw=0.5)
    ax.set_xlim(-1, 1)
    ax.set_xlabel(r'Spearman $\rho$  vs  per-target BEDROC (GBSA-locked)')
    ax.set_title(title, color=NAVY, fontweight='bold')
    ax.set_axisbelow(True)
    ax.xaxis.grid(True, color=GREY, alpha=0.5)

axes[0].text(0.02, 0.02, f'colored = p<0.05\n(unadjusted, n=8 targets)',
             transform=axes[0].transAxes, fontsize=8, color=GREYD,
             va='bottom', ha='left')
plt.tight_layout()

**Reading Figure 1:**

- **Protein side (left, NAVY):** a coherent physical cluster — active-site backbone
  RMSD, active-site RMSF, protein radius of gyration, active-site formal charge —
  correlates strongly with per-target BEDROC. Interpretation: **flexible, charged
  pockets are easier for GBSA to discriminate.** 30 ns MD samples multiple sub-states
  of a flexible pocket; that ensemble average is what MM-GBSA is scoring, so an
  averaging-friendly target rewards the method.

- **Ligand side (right, GOLD):** almost everything is grey (non-significant). Taking
  the median of a ligand feature across 30 different ligands per target smooths away
  the target-specific signal that a single-ligand experiment would carry. Only
  `lig_orient_autocorr_last` — how well a ligand keeps its starting orientation over
  30 ns — passes, and even that is arguably a pocket-rigidity readout via the ligand.

**So Q1 gives us the deployable diagnostic. Q2 is essentially noise.**

## Q3 — binder-vs-non-binder feature separation → BEDROC

> **Vocabulary reminder.** We call inactives *labelled non-binders* / *measured
> inactives* (pchembl < 5, experimentally measured), never *decoys* — this dataset
> contains no computationally-generated inactives.

Within each target, does a feature discriminate the labelled binders from the labelled
non-binders? If the discriminability magnitude correlates with per-target BEDROC, then
*targets whose active/inactive sets differ in that feature* are the ones GBSA ranks well.

We measure discriminability *two ways* because the numeric answer depends on the choice:

- **Cohen's d** = (mean binder − mean non-binder) / pooled_std. Parametric. Assumes
  approximately normal within-class distributions. One extreme value in either class
  moves d dramatically.
- **Cliff's δ** = P(random binder > random non-binder) − P(random binder < random
  non-binder). Rank-based. Bounded in [−1, 1]. Ignores the tails: a single outlier
  changes it by at most 1/(n_binder × n_non_binder).

For a small panel (n=8 targets × 10 binders × 20 non-binders per target) Cliff's δ
is the honest metric. Cohen's d is the enthusiastic one.

In [ ]:
def cliffs_delta(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    if len(a) < 3 or len(b) < 3: return np.nan
    gt = np.sum(a[:, None] > b[None, :])
    lt = np.sum(a[:, None] < b[None, :])
    return (gt - lt) / (len(a) * len(b))

def cohens_d(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    if len(a) < 3 or len(b) < 3: return np.nan
    pooled = np.sqrt((a.std()**2 + b.std()**2) / 2)
    return np.nan if pooled == 0 else (a.mean() - b.mean()) / pooled

FEATS_Q3 = ['lig_rg_mean_A', 'lig_buried_sasa_std_A2', 'n_hb_mean',
            'lig_MW', 'lig_TPSA', 'lig_LogP', 'lig_HBD', 'lig_HBA',
            'lig_orient_autocorr_last', 'lig_asphericity_mean']
FEATS_Q3 = [f for f in FEATS_Q3 if f in data.columns]

rows = []
for f in FEATS_Q3:
    cds, cls = {}, {}
    for t, g in data.groupby('target'):
        binder = g[g.is_active][f].dropna().values
        nonbinder = g[~g.is_active][f].dropna().values
        cds[t] = cohens_d(binder, nonbinder)
        cls[t] = cliffs_delta(binder, nonbinder)
    cds = pd.Series(cds); cls = pd.Series(cls)
    y1 = tgt_bedroc.reindex(cds.index).values
    m1 = np.isfinite(cds.values) & np.isfinite(y1)
    rho_d, p_d = spearmanr(np.abs(cds.values[m1]), y1[m1])
    m2 = np.isfinite(cls.values) & np.isfinite(y1)
    rho_c, p_c = spearmanr(np.abs(cls.values[m2]), y1[m2])
    rows.append({'feature': f, 'cohen_rho': rho_d, 'cohen_p': p_d,
                 'cliff_rho': rho_c, 'cliff_p': p_c})
q3 = pd.DataFrame(rows)

print('=== Q3 · discriminability (binders vs measured non-binders) → BEDROC ===')
print('Two metrics side by side — divergence flags outlier-driven inflation.\n')
display(q3.round(3))
q3.to_csv(DERIVED / 'target_footprint_q3_separation.csv', index=False)

In [ ]:
# Figure 2 — Cohen d vs Cliff's δ side-by-side for Q3 features
fig, ax = plt.subplots(figsize=(11, 5))

order = q3.sort_values('cohen_rho', key=abs, ascending=True).reset_index(drop=True)
ys = np.arange(len(order))
w = 0.38
ax.barh(ys - w/2, order['cohen_rho'], w, color=NAVY, edgecolor='white',
        label="Cohen's d (parametric)")
ax.barh(ys + w/2, order['cliff_rho'], w, color=GOLD, edgecolor='white',
        label="Cliff's δ (rank-based, robust)")

# annotate significance stars
for i, r in order.iterrows():
    for offset, val, p in [(-w/2, r['cohen_rho'], r['cohen_p']),
                            (+w/2, r['cliff_rho'], r['cliff_p'])]:
        if p < 0.05:
            ax.text(val + 0.02 * np.sign(val), i + offset, '*', ha='left' if val > 0 else 'right',
                    va='center', color=NAVY, fontsize=12, fontweight='bold')

ax.axvline(0, color=GREY, lw=0.6)
ax.set_yticks(ys); ax.set_yticklabels(order['feature'])
ax.set_xlim(-1, 1)
ax.set_xlabel(r'Spearman $\rho$ (|effect size| vs per-target BEDROC)')
ax.set_title("Q3 — the Cohen d ↔ Cliff's δ divergence: rank-based is honest",
             color=NAVY, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_axisbelow(True)
ax.xaxis.grid(True, color=GREY, alpha=0.5)
plt.tight_layout()

**Reading Figure 2:**

The top three Q3 features (`lig_rg_mean_A`, `lig_buried_sasa_std_A2`, `n_hb_mean`)
show a **large Cohen d correlation (ρ ≈ +0.7 to +0.88, all p < 0.05)** but shrink to
a **modest Cliff's δ correlation (ρ ≈ +0.4 to +0.52, all p > 0.15)** — meaning:

- The parametric signal is real, but it is being carried by one or two extreme targets
  where the binder/non-binder mean gap happens to be very large (Cohen d is unbounded).
- Under the rank-based metric, where a single extreme complex only changes δ by
  ≤ 1/(10 × 20) = 0.005, the panel-level correlation weakens sharply.

**Practical: at n = 8 targets, Cliff's δ is the honest metric.** Cohen d gave us a
false-positive-looking "binder discriminability predicts BEDROC" story.

This is a repeatable small-n lesson: when both effect sizes agree, you have signal.
When they diverge, the parametric one is inflated by outliers.

## Robustness — Q1 top hit under label permutation and Rg-MW collinearity check

Two sanity checks:

1. **Label permutation for Q1:** shuffle target ↔ BEDROC pairings 1000× and see
   how often we recover ρ ≥ +0.88 by chance. If p_perm ≪ 0.05, Q1 is robust.
2. **Rg ↔ MW collinearity:** an earlier version of this analysis "residualised"
   ligand features on lig_MW and reported a collapse of Q3 to ρ ≈ 0. But Rg
   correlates ~ρ = +0.84 with MW per target — residualising Rg on MW removes almost
   all the size information, leaving only orthogonal shape noise. The collapse is
   near-collinearity, not confounding. Document this trap here so no one repeats it.

In [ ]:
# --- (1) label-permutation test for Q1 top hit ---
top_prot = prot_corr.iloc[0]
top_feat = top_prot['feature']
obs_rho = float(top_prot['rho'])

rng = np.random.default_rng(20260909)
B = 2000
null_rhos = []
for _ in range(B):
    y_perm = rng.permutation(y)
    r, _ = spearmanr(per_t_med[top_feat].values, y_perm)
    null_rhos.append(r)
null_rhos = np.array(null_rhos)
p_perm = float(np.mean(np.abs(null_rhos) >= abs(obs_rho)))
print(f'Label-permutation test for Q1 top hit: {top_feat}')
print(f'  observed |ρ|     = {abs(obs_rho):.3f}')
print(f'  null mean(ρ)     = {null_rhos.mean():+.3f}')
print(f'  null 95%-|ρ|     = {np.percentile(np.abs(null_rhos), 95):.3f}')
print(f'  empirical p_perm = {p_perm:.4f}   (B = {B})')

# --- (2) Rg ↔ MW collinearity ---
print()
if 'lig_MW' in data.columns and 'lig_rg_mean_A' in data.columns:
    tmp = data.dropna(subset=['lig_MW', 'lig_rg_mean_A'])
    per_t_rgmw = tmp.groupby('target').apply(
        lambda g: spearmanr(g.lig_rg_mean_A, g.lig_MW)[0], include_groups=False)
    print(f'Rg ↔ MW collinearity — per-target Spearman ρ(lig_rg_mean_A, lig_MW):')
    print(per_t_rgmw.round(3).to_string())
    print(f'\n  Median across targets: {per_t_rgmw.median():+.3f}')
    print('  → Rg and MW are near-identical variables; residualising one on the other')
    print('    removes the physical size information, not a confound.')

In [ ]:
# Figure 3 — the story on one panel: three tests, three verdicts
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))

# (A) Q1 top hit — protein feature vs per-target BEDROC scatter
ax = axes[0]
x = per_t_med[top_feat].values
ax.scatter(x, y, color=NAVY, s=90, edgecolor='white', linewidth=0.7)
for xi, yi, name in zip(x, y, per_t_med.index):
    ax.annotate(name, (xi, yi), xytext=(4, 4), textcoords='offset points',
                fontsize=8, color=NAVY)
ax.set_xlabel(f'{top_feat}   (per-target median)')
ax.set_ylabel('per-target BEDROC (GBSA-locked)')
ax.set_title(f"(A) Q1 · robust\nρ={obs_rho:+.2f}, p={float(top_prot['p']):.3f}, p_perm={p_perm:.3f}",
             color=NAVY, fontsize=10)
ax.set_axisbelow(True); ax.grid(True, color=GREY, alpha=0.3)

# (B) Q2 example — a ligand-side aggregate that looks flat
lig_flat = lig_corr.iloc[len(lig_corr)//2]  # median-|ρ| ligand feature
x2 = per_t_med[lig_flat['feature']].values
ax = axes[1]
ax.scatter(x2, y, color=GOLD, s=90, edgecolor='white', linewidth=0.7)
for xi, yi, name in zip(x2, y, per_t_med.index):
    ax.annotate(name, (xi, yi), xytext=(4, 4), textcoords='offset points',
                fontsize=8, color=NAVY)
ax.set_xlabel(f'{lig_flat["feature"]}   (per-target median)')
ax.set_ylabel('per-target BEDROC (GBSA-locked)')
ax.set_title(f"(B) Q2 · noise\nρ={float(lig_flat['rho']):+.2f}, p={float(lig_flat['p']):.3f}",
             color=NAVY, fontsize=10)
ax.set_axisbelow(True); ax.grid(True, color=GREY, alpha=0.3)

# (C) Q3 divergence — Cohen vs Cliff for top 3 features
ax = axes[2]
topq3 = q3.sort_values('cohen_rho', key=abs, ascending=False).head(3)
xs = np.arange(len(topq3)); w = 0.38
ax.bar(xs - w/2, topq3['cohen_rho'], w, color=NAVY, edgecolor='white', label="Cohen d")
ax.bar(xs + w/2, topq3['cliff_rho'], w, color=GOLD, edgecolor='white', label="Cliff's δ")
ax.set_xticks(xs)
ax.set_xticklabels([str(f).replace('lig_', '') for f in topq3['feature']], rotation=25, ha='right', fontsize=8)
ax.set_ylabel(r'Spearman $\rho$  vs BEDROC')
ax.set_ylim(-0.2, 1.05)
ax.axhline(0, color=GREY, lw=0.6)
ax.set_title("(C) Q3 · Cohen inflated; Cliff honest", color=NAVY, fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.3)

plt.tight_layout()

## Verdict — the clean story from three univariate tests

| Question | Metric | ρ vs per-target BEDROC | Verdict |
|---|---|---:|---|
| **Q1** — protein-side aggregate | Spearman (median vs BEDROC) | **+0.88 (p=0.004, p_perm≈0.004)** | ✅ robust deployable signal |
| **Q2** — ligand-side aggregate | Spearman (median vs BEDROC) | ≤ +0.65 (1/42 with p<0.05) | ✗ essentially noise |
| **Q3a** — binder / non-binder separation | **Cohen d** (parametric) | +0.88 | ⚠ apparent — inflated |
| **Q3b** — binder / non-binder separation | **Cliff's δ** (rank-based) | +0.43 (p=0.29) | ✗ collapses under rank |

**Take-home lessons:**

1. **Protein flexibility is the honest predictor.** Active-site backbone RMSD,
   active-site RMSF, protein Rg, and active-site formal charge together form a
   coherent physical cluster that predicts per-target BEDROC. All four are direct
   target properties, so they cannot be confounded by properties of the labelled
   ligand set.
2. **Ligand-side aggregates are noise at target level.** The median across 30
   different ligands smooths away target-specific signal. Only
   `lig_orient_autocorr_last` — a pocket-rigidity readout via the ligand — carries.
3. **Binder / non-binder discriminability looks predictive but is not** at n = 8.
   Cohen d inflates the panel-level correlation; Cliff's δ says the effect is real
   but too weak to detect on this sample size.
4. **Rank-based effect sizes are mandatory for small panels.** Cohen d assumes
   distribution shape; a single extreme target can drive the panel result. Cliff's
   δ is bounded and rank-based — it never says more than it should.
5. **The Rg ↔ MW collinearity trap:** residualising a ligand-size feature on MW
   removes almost all the physical signal, because Rg is essentially MW in disguise
   (per-target ρ = +0.84). This is *collinearity*, not confounding. Do not treat
   MW-differences between binders and measured non-binders as a confound to control
   away — they are the natural chemistry of the ChEMBL non-binder sample.

**Deployable diagnostic (Q1 protein cluster):** given a new target we can compute
the four Q1 features from a **short apo-protein MD (≤ 5 ns) plus the delivered PDB**,
no ligand needed. That predicts, ahead of any GBSA rescoring campaign, whether
MMGBSA is likely to rank ligands well on that target. The 18 additional newbench-27
targets we're preparing to run are the natural validation set.

**Written files:**

```
data/external/gbsa-study/data/derived/target_footprint_q1_protein.csv
data/external/gbsa-study/data/derived/target_footprint_q2_ligand.csv
data/external/gbsa-study/data/derived/target_footprint_q3_separation.csv
figures/34_target_footprint_diagnostic_fig{1..3}.png
```

In [ ]:
# --- export every figure produced in this notebook ---
FIGURES.mkdir(parents=True, exist_ok=True)
figs = list(globals().get('_SAVED_FIGS', []))
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs: figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f'{NB_STEM}_fig{i}.png'
    try:
        fig.savefig(out, bbox_inches='tight', dpi=200, facecolor=CREAM)
        saved.append(out.name)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
print(f'saved {len(saved)} figures: {saved}')